# Statistics for Machine Learning

## Why Statistics Matters for ML

Statistics is the **mathematical foundation** of machine learning. Understanding statistics helps you:

| Skill | ML Application |
|-------|---------------|
| Descriptive Stats | Understand your data before modeling |
| Probability | Understand model predictions and uncertainty |
| Distributions | Choose appropriate models and preprocessing |
| Hypothesis Testing | Validate model improvements statistically |
| Sampling | Create proper train/test splits |

### Topics Covered

1. **Descriptive Statistics** - Summarizing data
2. **Probability Fundamentals** - Probability rules and Bayes theorem
3. **Probability Distributions** - Normal, binomial, and more
4. **Central Limit Theorem** - Why it enables statistical inference
5. **Hypothesis Testing** - A/B testing and model comparison
6. **Correlation & Covariance** - Measuring relationships
7. **Statistical Tests for ML** - Practical applications

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Configure plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print('Libraries loaded successfully!')
print(f'NumPy: {np.__version__}')
print(f'SciPy stats module ready')

In [ ]:
# Create sample dataset for examples
n_samples = 500

# Simulate employee data
data = {
    'age': np.random.normal(35, 10, n_samples).clip(22, 65).astype(int),
    'salary': np.random.lognormal(10.8, 0.4, n_samples).astype(int),
    'experience_years': np.random.exponential(5, n_samples).clip(0, 30).astype(int),
    'performance_score': np.random.normal(75, 12, n_samples).clip(0, 100),
    'satisfaction': np.random.beta(5, 2, n_samples) * 10,  # Skewed right
    'department': np.random.choice(['Engineering', 'Sales', 'Marketing', 'HR', 'Finance'], n_samples),
    'promoted': np.random.binomial(1, 0.2, n_samples),  # 20% promotion rate
}

df = pd.DataFrame(data)

# Add some correlation: higher experience -> higher salary
df['salary'] = df['salary'] + df['experience_years'] * 2000

print(f'Dataset shape: {df.shape}')
df.head()

---

## 1. Descriptive Statistics

Descriptive statistics **summarize** and **describe** the main features of a dataset.

### Types of Descriptive Statistics

| Type | Measures | Purpose |
|------|----------|--------|
| **Central Tendency** | Mean, Median, Mode | Where is the center? |
| **Dispersion/Spread** | Variance, Std Dev, Range, IQR | How spread out? |
| **Shape** | Skewness, Kurtosis | What's the distribution shape? |
| **Position** | Percentiles, Quartiles | Where do values fall? |

### 1.1 Measures of Central Tendency

**Mean** ($\bar{x}$): Average of all values
$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$

**Median**: Middle value when sorted (robust to outliers)

**Mode**: Most frequent value

In [ ]:
# Central tendency measures
salary = df['salary']

print('=== Measures of Central Tendency ===')
print(f'Mean salary:   ${salary.mean():,.2f}')
print(f'Median salary: ${salary.median():,.2f}')
print(f'Mode salary:   ${salary.mode().values[0]:,.2f}')

# When mean != median, data is skewed
if salary.mean() > salary.median():
    print('\n📊 Mean > Median: Right-skewed distribution (outliers pull mean up)')
elif salary.mean() < salary.median():
    print('\n📊 Mean < Median: Left-skewed distribution (outliers pull mean down)')
else:
    print('\n📊 Mean ≈ Median: Symmetric distribution')

In [ ]:
# Visualize mean vs median
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Salary distribution (skewed)
axes[0].hist(df['salary'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(df['salary'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${df["salary"].mean():,.0f}')
axes[0].axvline(df['salary'].median(), color='green', linestyle='-', linewidth=2, label=f'Median: ${df["salary"].median():,.0f}')
axes[0].set_title('Salary Distribution (Right-Skewed)', fontweight='bold')
axes[0].set_xlabel('Salary ($)')
axes[0].legend()

# Performance distribution (more symmetric)
axes[1].hist(df['performance_score'], bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(df['performance_score'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["performance_score"].mean():.1f}')
axes[1].axvline(df['performance_score'].median(), color='green', linestyle='-', linewidth=2, label=f'Median: {df["performance_score"].median():.1f}')
axes[1].set_title('Performance Score (More Symmetric)', fontweight='bold')
axes[1].set_xlabel('Score')
axes[1].legend()

plt.tight_layout()
plt.show()

print('💡 ML Tip: Use median for skewed data, mean for symmetric data.')

### 1.2 Measures of Dispersion (Spread)

**Variance** ($\sigma^2$): Average squared deviation from mean
$$\sigma^2 = \frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2$$

**Standard Deviation** ($\sigma$): Square root of variance (same units as data)
$$\sigma = \sqrt{\sigma^2}$$

**Range**: Max - Min

**IQR (Interquartile Range)**: Q3 - Q1 (robust to outliers)

In [ ]:
# Measures of dispersion
print('=== Measures of Dispersion ===')
print(f'\nSalary:')
print(f'  Variance:  ${salary.var():,.2f}')
print(f'  Std Dev:   ${salary.std():,.2f}')
print(f'  Range:     ${salary.max() - salary.min():,.2f}')
print(f'  IQR:       ${salary.quantile(0.75) - salary.quantile(0.25):,.2f}')

# Coefficient of Variation (CV) - relative measure of spread
cv = (salary.std() / salary.mean()) * 100
print(f'  CV:        {cv:.1f}%')

print('\n💡 ML Tip: High CV (>30%) indicates high variability - consider scaling features.')

In [ ]:
# Compare dispersion across features
numeric_cols = ['age', 'salary', 'experience_years', 'performance_score', 'satisfaction']

dispersion_df = pd.DataFrame({
    'Feature': numeric_cols,
    'Mean': [df[col].mean() for col in numeric_cols],
    'Std Dev': [df[col].std() for col in numeric_cols],
    'CV (%)': [(df[col].std() / df[col].mean() * 100) for col in numeric_cols],
    'IQR': [df[col].quantile(0.75) - df[col].quantile(0.25) for col in numeric_cols]
})

print('Dispersion Comparison:')
print(dispersion_df.round(2).to_string(index=False))

### 1.3 Measures of Shape

**Skewness**: Measures asymmetry of distribution
- Skewness = 0: Symmetric
- Skewness > 0: Right-skewed (tail extends right)
- Skewness < 0: Left-skewed (tail extends left)

**Kurtosis**: Measures "tailedness" (extreme values)
- Kurtosis = 3: Normal distribution (mesokurtic)
- Kurtosis > 3: Heavy tails, more outliers (leptokurtic)
- Kurtosis < 3: Light tails, fewer outliers (platykurtic)

In [ ]:
# Calculate skewness and kurtosis
print('=== Shape Statistics ===')
for col in numeric_cols:
    skew = df[col].skew()
    kurt = df[col].kurtosis()  # Fisher's kurtosis (normal = 0)
    
    # Interpret skewness
    if abs(skew) < 0.5:
        skew_type = 'symmetric'
    elif skew > 0:
        skew_type = 'right-skewed'
    else:
        skew_type = 'left-skewed'
    
    print(f'{col:20s}: skew={skew:6.2f} ({skew_type:12s}), kurtosis={kurt:6.2f}')

In [ ]:
# Visualize different distribution shapes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Right-skewed (salary)
sns.histplot(df['salary'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title(f'Right-Skewed\nSkew: {df["salary"].skew():.2f}', fontweight='bold')
axes[0].set_xlabel('Salary')

# Approximately symmetric (performance)
sns.histplot(df['performance_score'], kde=True, ax=axes[1], color='forestgreen')
axes[1].set_title(f'Approximately Symmetric\nSkew: {df["performance_score"].skew():.2f}', fontweight='bold')
axes[1].set_xlabel('Performance Score')

# Left-skewed (satisfaction - beta distribution)
sns.histplot(df['satisfaction'], kde=True, ax=axes[2], color='coral')
axes[2].set_title(f'Left-Skewed\nSkew: {df["satisfaction"].skew():.2f}', fontweight='bold')
axes[2].set_xlabel('Satisfaction')

plt.tight_layout()
plt.show()

print('💡 ML Tip: Skewed features may need log/sqrt transformation before modeling.')

### 1.4 Percentiles and Quartiles

**Percentile**: Value below which a given percentage of data falls
- 25th percentile (Q1): First quartile
- 50th percentile (Q2): Median
- 75th percentile (Q3): Third quartile

**Box Plot Anatomy**:
```
    |----[====|====]----|
    ↑    ↑    ↑    ↑    ↑
  Min   Q1  Median Q3  Max (or whiskers)
```

In [ ]:
# Calculate percentiles
print('=== Salary Percentiles ===')
percentiles = [10, 25, 50, 75, 90, 95, 99]

for p in percentiles:
    value = df['salary'].quantile(p/100)
    print(f'  {p:2d}th percentile: ${value:>10,.2f}')

print(f'\n  IQR (Q3-Q1):     ${df["salary"].quantile(0.75) - df["salary"].quantile(0.25):>10,.2f}')

In [ ]:
# Box plot with percentile annotations
fig, ax = plt.subplots(figsize=(12, 6))

bp = ax.boxplot(df['salary'], vert=False, widths=0.5, patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')

# Add percentile annotations
q1 = df['salary'].quantile(0.25)
q2 = df['salary'].quantile(0.50)
q3 = df['salary'].quantile(0.75)

ax.annotate(f'Q1\n${q1:,.0f}', xy=(q1, 1), xytext=(q1, 1.3),
            ha='center', fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate(f'Median\n${q2:,.0f}', xy=(q2, 1), xytext=(q2, 1.4),
            ha='center', fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate(f'Q3\n${q3:,.0f}', xy=(q3, 1), xytext=(q3, 1.3),
            ha='center', fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_xlabel('Salary ($)', fontsize=12)
ax.set_title('Salary Distribution - Box Plot with Quartiles', fontweight='bold', fontsize=14)
ax.set_ylim(0.5, 1.6)

plt.tight_layout()
plt.show()

In [ ]:
# Quick summary with pandas describe()
print('=== Full Statistical Summary ===')
print(df[numeric_cols].describe().round(2))

print('\n💡 ML Tip: df.describe() gives you quick EDA for all numeric columns.')

---

## 2. Probability Fundamentals

Probability is the **language of uncertainty** in ML. Every prediction has uncertainty.

### Key Probability Concepts

| Concept | Definition | ML Application |
|---------|------------|---------------|
| $P(A)$ | Probability of event A | Model prediction confidence |
| $P(A \cap B)$ | Probability of A AND B | Joint feature probability |
| $P(A \cup B)$ | Probability of A OR B | Combined event probability |
| $P(A|B)$ | Probability of A given B | Conditional classification |

### Probability Rules

1. **Range**: $0 \leq P(A) \leq 1$
2. **Complement**: $P(A') = 1 - P(A)$
3. **Addition**: $P(A \cup B) = P(A) + P(B) - P(A \cap B)$
4. **Multiplication**: $P(A \cap B) = P(A) \cdot P(B|A)$

In [ ]:
# Basic probability calculations
print('=== Basic Probability Examples ===')

# P(promoted)
p_promoted = df['promoted'].mean()
print(f'P(Promoted) = {p_promoted:.3f} ({p_promoted*100:.1f}%)')

# P(Engineering)
p_eng = (df['department'] == 'Engineering').mean()
print(f'P(Engineering) = {p_eng:.3f} ({p_eng*100:.1f}%)')

# P(high salary) - above median
p_high_salary = (df['salary'] > df['salary'].median()).mean()
print(f'P(High Salary) = {p_high_salary:.3f} ({p_high_salary*100:.1f}%)')

In [ ]:
# Conditional probability: P(A|B) = P(A and B) / P(B)
print('=== Conditional Probability ===')

# P(Promoted | Engineering)
eng_mask = df['department'] == 'Engineering'
p_promoted_given_eng = df.loc[eng_mask, 'promoted'].mean()
print(f'P(Promoted | Engineering) = {p_promoted_given_eng:.3f}')

# Compare with other departments
print('\nPromotion rates by department:')
promo_by_dept = df.groupby('department')['promoted'].mean().sort_values(ascending=False)
for dept, rate in promo_by_dept.items():
    print(f'  {dept:12s}: {rate:.3f} ({rate*100:.1f}%)')

In [ ]:
# Joint probability: P(A and B)
print('=== Joint Probability ===')

# P(Engineering AND Promoted)
p_eng_and_promoted = ((df['department'] == 'Engineering') & (df['promoted'] == 1)).mean()
print(f'P(Engineering AND Promoted) = {p_eng_and_promoted:.3f}')

# Create contingency table
contingency = pd.crosstab(df['department'], df['promoted'], margins=True, normalize='all')
print('\nJoint Probability Table (normalized):')
print(contingency.round(3))

### 2.1 Bayes' Theorem

Bayes' theorem relates conditional probabilities:

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

**ML Application**: Naive Bayes classifier uses this theorem!

**Components**:
- $P(A|B)$: **Posterior** - probability after seeing evidence
- $P(A)$: **Prior** - initial probability
- $P(B|A)$: **Likelihood** - probability of evidence given hypothesis
- $P(B)$: **Evidence** - total probability of evidence

In [ ]:
# Bayes' Theorem Example
# Question: If someone is promoted, what's the probability they're in Engineering?

print('=== Bayes Theorem Example ===')
print('Question: P(Engineering | Promoted) = ?')
print()

# Known probabilities
p_eng = (df['department'] == 'Engineering').mean()  # P(Engineering)
p_promoted = df['promoted'].mean()  # P(Promoted)
p_promoted_given_eng = df.loc[df['department'] == 'Engineering', 'promoted'].mean()  # P(Promoted|Engineering)

print(f'P(Engineering) = {p_eng:.3f}')
print(f'P(Promoted) = {p_promoted:.3f}')
print(f'P(Promoted | Engineering) = {p_promoted_given_eng:.3f}')

# Apply Bayes' theorem
p_eng_given_promoted = (p_promoted_given_eng * p_eng) / p_promoted
print(f'\nP(Engineering | Promoted) = {p_eng_given_promoted:.3f}')

# Verify with direct calculation
p_eng_given_promoted_direct = df.loc[df['promoted'] == 1, 'department'].value_counts(normalize=True).get('Engineering', 0)
print(f'Direct calculation: {p_eng_given_promoted_direct:.3f}')

In [ ]:
# Visualize conditional probabilities
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Promotion rate by department
promo_rates = df.groupby('department')['promoted'].mean().sort_values()
colors = ['green' if r > df['promoted'].mean() else 'gray' for r in promo_rates.values]
axes[0].barh(promo_rates.index, promo_rates.values, color=colors, edgecolor='black')
axes[0].axvline(df['promoted'].mean(), color='red', linestyle='--', label=f'Overall: {df["promoted"].mean():.1%}')
axes[0].set_xlabel('Promotion Rate')
axes[0].set_title('P(Promoted | Department)', fontweight='bold', fontsize=12)
axes[0].legend()

# Format as percentage
for i, (dept, rate) in enumerate(promo_rates.items()):
    axes[0].text(rate + 0.01, i, f'{rate:.1%}', va='center')

# Department distribution among promoted
promoted_dept = df.loc[df['promoted'] == 1, 'department'].value_counts(normalize=True)
axes[1].pie(promoted_dept.values, labels=promoted_dept.index, autopct='%1.1f%%',
            colors=plt.cm.Set3.colors[:len(promoted_dept)])
axes[1].set_title('P(Department | Promoted)', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

print('💡 ML Tip: Naive Bayes assumes feature independence to simplify P(features|class).')

---

## 3. Probability Distributions

A probability distribution describes how likely different values are.

### Types of Distributions

| Type | Examples | Use Case |
|------|----------|----------|
| **Discrete** | Binomial, Poisson | Countable outcomes (clicks, counts) |
| **Continuous** | Normal, Exponential | Measurable values (height, time) |

### Key Distributions for ML

1. **Normal (Gaussian)** - Most important! Many algorithms assume normality
2. **Binomial** - Binary outcomes (click/no-click)
3. **Uniform** - Equal probability (random sampling)
4. **Exponential** - Time between events

### 3.1 Normal Distribution (Gaussian)

The **most important** distribution in statistics and ML.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

**Parameters**:
- $\mu$ (mu): Mean - center of distribution
- $\sigma$ (sigma): Standard deviation - spread

**68-95-99.7 Rule**:
- 68% of data within 1σ of mean
- 95% of data within 2σ of mean
- 99.7% of data within 3σ of mean

In [ ]:
# Normal distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Standard normal distribution
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x, 0, 1)  # mean=0, std=1

axes[0].plot(x, y, 'b-', linewidth=2, label='N(0, 1)')
axes[0].fill_between(x, y, where=(x >= -1) & (x <= 1), alpha=0.3, color='blue', label='68% (1σ)')
axes[0].fill_between(x, y, where=(x >= -2) & (x <= 2), alpha=0.2, color='green', label='95% (2σ)')
axes[0].set_title('Standard Normal Distribution', fontweight='bold')
axes[0].set_xlabel('Z-score')
axes[0].set_ylabel('Probability Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Compare different normal distributions
for mu, sigma, color in [(0, 1, 'blue'), (0, 2, 'red'), (2, 1, 'green')]:
    y = stats.norm.pdf(x, mu, sigma)
    axes[1].plot(x, y, color=color, linewidth=2, label=f'μ={mu}, σ={sigma}')

axes[1].set_title('Effect of μ and σ', fontweight='bold')
axes[1].set_xlabel('x')
axes[1].set_ylabel('Probability Density')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Z-scores: Standardizing data
# Z = (x - μ) / σ

print('=== Z-Scores (Standardization) ===')
print('Z-score tells you how many standard deviations from the mean.')
print()

# Calculate z-scores for salary
salary_mean = df['salary'].mean()
salary_std = df['salary'].std()
df['salary_zscore'] = (df['salary'] - salary_mean) / salary_std

print(f'Salary Mean: ${salary_mean:,.2f}')
print(f'Salary Std:  ${salary_std:,.2f}')
print()

# Interpret some z-scores
example_salaries = [salary_mean, salary_mean + salary_std, salary_mean + 2*salary_std]
for sal in example_salaries:
    z = (sal - salary_mean) / salary_std
    percentile = stats.norm.cdf(z) * 100
    print(f'Salary ${sal:>10,.0f} → Z={z:5.2f} → {percentile:5.1f}th percentile')

print('\n💡 ML Tip: StandardScaler in sklearn does exactly this transformation!')

In [ ]:
# Test if data is normally distributed
print('=== Normality Tests ===')

for col in ['performance_score', 'salary', 'satisfaction']:
    # Shapiro-Wilk test (best for n < 5000)
    stat, p_value = stats.shapiro(df[col].sample(min(len(df), 500)))
    
    # D'Agostino-Pearson test
    stat2, p_value2 = stats.normaltest(df[col])
    
    is_normal = 'Yes' if p_value > 0.05 else 'No'
    print(f'{col:20s}: Shapiro p={p_value:.4f}, Normal? {is_normal}')

print('\n💡 ML Tip: Many ML algorithms work better with normally distributed features.')
print('   If not normal, consider: log transform, Box-Cox, or quantile transform.')

### 3.2 Other Important Distributions

In [ ]:
# Visualize common distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Binomial - number of successes in n trials
n, p = 20, 0.3
x_binom = np.arange(0, n+1)
y_binom = stats.binom.pmf(x_binom, n, p)
axes[0, 0].bar(x_binom, y_binom, color='steelblue', edgecolor='black')
axes[0, 0].set_title(f'Binomial Distribution\nn={n}, p={p}', fontweight='bold')
axes[0, 0].set_xlabel('Number of Successes')
axes[0, 0].set_ylabel('Probability')
axes[0, 0].text(0.95, 0.95, 'Use: Binary outcomes\n(clicks, conversions)', 
                transform=axes[0, 0].transAxes, ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 2. Poisson - count of events in fixed interval
lambda_param = 5
x_pois = np.arange(0, 20)
y_pois = stats.poisson.pmf(x_pois, lambda_param)
axes[0, 1].bar(x_pois, y_pois, color='forestgreen', edgecolor='black')
axes[0, 1].set_title(f'Poisson Distribution\nλ={lambda_param}', fontweight='bold')
axes[0, 1].set_xlabel('Count')
axes[0, 1].set_ylabel('Probability')
axes[0, 1].text(0.95, 0.95, 'Use: Count data\n(visits/hour, errors/day)', 
                transform=axes[0, 1].transAxes, ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 3. Exponential - time between events
x_exp = np.linspace(0, 10, 1000)
for rate in [0.5, 1, 2]:
    y_exp = stats.expon.pdf(x_exp, scale=1/rate)
    axes[1, 0].plot(x_exp, y_exp, linewidth=2, label=f'λ={rate}')
axes[1, 0].set_title('Exponential Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_ylabel('Probability Density')
axes[1, 0].legend()
axes[1, 0].text(0.95, 0.95, 'Use: Time until event\n(customer wait time)', 
                transform=axes[1, 0].transAxes, ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 4. Uniform - equal probability
x_unif = np.linspace(-0.5, 1.5, 1000)
y_unif = stats.uniform.pdf(x_unif, 0, 1)
axes[1, 1].plot(x_unif, y_unif, 'b-', linewidth=2)
axes[1, 1].fill_between(x_unif, y_unif, alpha=0.3)
axes[1, 1].set_title('Uniform Distribution\na=0, b=1', fontweight='bold')
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('Probability Density')
axes[1, 1].set_ylim(0, 1.5)
axes[1, 1].text(0.95, 0.95, 'Use: Random sampling\n(np.random.uniform)', 
                transform=axes[1, 1].transAxes, ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

---

## 4. Central Limit Theorem (CLT)

The **Central Limit Theorem** is the foundation of statistical inference.

### The CLT Statement

> The distribution of **sample means** approaches a normal distribution as sample size increases, 
> regardless of the original population distribution.

**Key Points**:
- Works for any distribution with finite mean and variance
- Sample size n ≥ 30 is usually sufficient
- Sample mean: $\bar{X} \sim N(\mu, \frac{\sigma}{\sqrt{n}})$

**Standard Error** = $\frac{\sigma}{\sqrt{n}}$ (decreases as sample size increases)

In [ ]:
# Demonstrate CLT with different distributions
np.random.seed(42)

# Create a clearly non-normal population (exponential)
population = np.random.exponential(scale=2, size=100000)

# Take many samples and calculate means
sample_sizes = [5, 30, 100]
n_samples = 1000

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Original population
axes[0, 0].hist(population, bins=50, density=True, alpha=0.7, color='gray')
axes[0, 0].set_title('Population (Exponential)\nSkewed!', fontweight='bold')
axes[0, 0].set_xlabel('Value')

# Distribution of sample means for different sample sizes
for idx, n in enumerate(sample_sizes):
    sample_means = [np.random.choice(population, n).mean() for _ in range(n_samples)]
    
    # Top row: histograms
    axes[0, idx].hist(sample_means, bins=30, density=True, alpha=0.7, color='steelblue')
    
    # Overlay normal curve
    x = np.linspace(min(sample_means), max(sample_means), 100)
    expected_std = population.std() / np.sqrt(n)
    axes[0, idx].plot(x, stats.norm.pdf(x, population.mean(), expected_std), 'r-', linewidth=2)
    axes[0, idx].set_title(f'Sample Means (n={n})\nSkew={pd.Series(sample_means).skew():.2f}', fontweight='bold')
    axes[0, idx].set_xlabel('Sample Mean')
    
    # Bottom row: Q-Q plots
    stats.probplot(sample_means, dist='norm', plot=axes[1, idx])
    axes[1, idx].set_title(f'Q-Q Plot (n={n})', fontweight='bold')

plt.tight_layout()
plt.show()

print('💡 Observation: As n increases, sample means become more normally distributed!')

In [ ]:
# Standard Error demonstration
print('=== Standard Error of the Mean ===')
print('Standard Error = σ / √n')
print('\nAs sample size increases, standard error decreases:')
print()

pop_std = population.std()
for n in [10, 30, 100, 500, 1000]:
    se = pop_std / np.sqrt(n)
    print(f'  n={n:4d}: SE = {se:.4f}')

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))

sample_sizes = np.arange(10, 501, 10)
standard_errors = pop_std / np.sqrt(sample_sizes)

ax.plot(sample_sizes, standard_errors, 'b-', linewidth=2)
ax.fill_between(sample_sizes, 0, standard_errors, alpha=0.3)
ax.set_xlabel('Sample Size (n)', fontsize=12)
ax.set_ylabel('Standard Error', fontsize=12)
ax.set_title('Standard Error Decreases with Sample Size', fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3)

# Annotate diminishing returns
ax.axvline(30, color='red', linestyle='--', alpha=0.7)
ax.text(35, standard_errors[2], 'n=30\n(rule of thumb)', fontsize=10)

plt.tight_layout()
plt.show()

print('\n💡 ML Tip: Diminishing returns after n≈30-100. Larger samples = more precision but higher cost.')

### 4.1 Confidence Intervals

A **confidence interval** gives a range of plausible values for a population parameter.

$$CI = \bar{x} \pm z_{\alpha/2} \cdot \frac{s}{\sqrt{n}}$$

| Confidence Level | Z-value |
|------------------|--------|
| 90% | 1.645 |
| 95% | 1.96 |
| 99% | 2.576 |

In [ ]:
# Calculate confidence intervals
print('=== Confidence Intervals for Mean Salary ===')

sample = df['salary']
n = len(sample)
mean = sample.mean()
se = sample.std() / np.sqrt(n)

for confidence in [0.90, 0.95, 0.99]:
    z = stats.norm.ppf((1 + confidence) / 2)
    margin = z * se
    ci_low = mean - margin
    ci_high = mean + margin
    
    print(f'{confidence*100:.0f}% CI: ${ci_low:,.2f} to ${ci_high:,.2f} (margin: ±${margin:,.2f})')

print(f'\nSample mean: ${mean:,.2f}')
print(f'Sample size: {n}')

In [ ]:
# Visualize confidence interval
fig, ax = plt.subplots(figsize=(12, 5))

# Multiple samples to show CI coverage
n_simulations = 50
sample_size = 50
true_mean = df['salary'].mean()

covered = 0
for i in range(n_simulations):
    sample = df['salary'].sample(sample_size, replace=True)
    mean = sample.mean()
    se = sample.std() / np.sqrt(sample_size)
    ci_low = mean - 1.96 * se
    ci_high = mean + 1.96 * se
    
    # Check if true mean is covered
    if ci_low <= true_mean <= ci_high:
        color = 'blue'
        covered += 1
    else:
        color = 'red'
    
    ax.plot([ci_low, ci_high], [i, i], color=color, linewidth=2)
    ax.plot(mean, i, 'o', color=color, markersize=4)

# True mean
ax.axvline(true_mean, color='green', linewidth=3, linestyle='--', label=f'True Mean: ${true_mean:,.0f}')

ax.set_xlabel('Salary ($)', fontsize=12)
ax.set_ylabel('Sample #', fontsize=12)
ax.set_title(f'95% Confidence Intervals\n({covered}/{n_simulations} = {covered/n_simulations*100:.0f}% contain true mean)', 
             fontweight='bold', fontsize=14)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

print('💡 ML Tip: 95% CI means if we repeat sampling, 95% of CIs will contain true mean.')

---

## 5. Hypothesis Testing

Hypothesis testing helps us make **data-driven decisions** about population parameters.

### The Hypothesis Testing Framework

1. **State hypotheses**:
   - $H_0$ (Null): No effect, no difference (status quo)
   - $H_1$ (Alternative): There IS an effect/difference

2. **Choose significance level** ($\alpha$): Usually 0.05

3. **Calculate test statistic** and p-value

4. **Make decision**:
   - p-value < α → Reject $H_0$
   - p-value ≥ α → Fail to reject $H_0$

### ML Applications
- A/B testing (is new model better?)
- Feature selection (is feature significant?)
- Model comparison (is difference real or chance?)

In [ ]:
# T-test: Compare means of two groups
print('=== T-Test: Engineering vs Sales Salary ===')
print()

eng_salary = df[df['department'] == 'Engineering']['salary']
sales_salary = df[df['department'] == 'Sales']['salary']

print(f'Engineering: n={len(eng_salary)}, mean=${eng_salary.mean():,.0f}, std=${eng_salary.std():,.0f}')
print(f'Sales:       n={len(sales_salary)}, mean=${sales_salary.mean():,.0f}, std=${sales_salary.std():,.0f}')
print()

# Independent samples t-test
t_stat, p_value = stats.ttest_ind(eng_salary, sales_salary)

print(f'H₀: Mean salary of Engineering = Mean salary of Sales')
print(f'H₁: Mean salary of Engineering ≠ Mean salary of Sales')
print()
print(f'T-statistic: {t_stat:.4f}')
print(f'P-value:     {p_value:.4f}')
print()

alpha = 0.05
if p_value < alpha:
    print(f'✅ p={p_value:.4f} < α={alpha} → Reject H₀')
    print('   There IS a significant difference in salaries.')
else:
    print(f'❌ p={p_value:.4f} ≥ α={alpha} → Fail to reject H₀')
    print('   No significant difference detected.')

In [ ]:
# Visualize the t-test
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot comparison
df_subset = df[df['department'].isin(['Engineering', 'Sales'])]
sns.boxplot(x='department', y='salary', data=df_subset, ax=axes[0], palette='Set2')
axes[0].set_title(f'Salary Comparison\np-value = {p_value:.4f}', fontweight='bold')
axes[0].set_ylabel('Salary ($)')

# Distribution comparison
sns.kdeplot(eng_salary, ax=axes[1], label='Engineering', fill=True, alpha=0.5)
sns.kdeplot(sales_salary, ax=axes[1], label='Sales', fill=True, alpha=0.5)
axes[1].axvline(eng_salary.mean(), color='blue', linestyle='--', label=f'Eng Mean: ${eng_salary.mean():,.0f}')
axes[1].axvline(sales_salary.mean(), color='orange', linestyle='--', label=f'Sales Mean: ${sales_salary.mean():,.0f}')
axes[1].set_title('Salary Distributions', fontweight='bold')
axes[1].set_xlabel('Salary ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.1 Types of Errors

| | H₀ True | H₀ False |
|---|---------|----------|
| **Reject H₀** | Type I Error (α) | Correct (Power) |
| **Fail to Reject H₀** | Correct | Type II Error (β) |

- **Type I Error (False Positive)**: Reject H₀ when it's true (probability = α)
- **Type II Error (False Negative)**: Fail to reject H₀ when it's false (probability = β)
- **Power** = 1 - β (probability of detecting a real effect)

In [ ]:
# Effect size: Cohen's d
# Measures practical significance (not just statistical significance)

def cohens_d(group1, group2):
    """Calculate Cohen's d for effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    
    return (group1.mean() - group2.mean()) / pooled_std

d = cohens_d(eng_salary, sales_salary)

print('=== Effect Size: Cohen\'s d ===')
print(f"Cohen's d: {d:.3f}")
print()
print('Interpretation:')
print('  |d| < 0.2:  Small effect')
print('  0.2 ≤ |d| < 0.8: Medium effect')
print('  |d| ≥ 0.8: Large effect')
print()

if abs(d) < 0.2:
    print(f'  → |d|={abs(d):.3f} indicates a SMALL effect')
elif abs(d) < 0.8:
    print(f'  → |d|={abs(d):.3f} indicates a MEDIUM effect')
else:
    print(f'  → |d|={abs(d):.3f} indicates a LARGE effect')

print('\n💡 ML Tip: Statistical significance ≠ practical significance. Always check effect size!')

In [ ]:
# Chi-square test: Independence of categorical variables
print('=== Chi-Square Test: Department vs Promotion ===')
print('Is promotion rate independent of department?')
print()

# Create contingency table
contingency = pd.crosstab(df['department'], df['promoted'])
print('Contingency Table:')
print(contingency)
print()

# Chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f'Chi-square statistic: {chi2:.4f}')
print(f'Degrees of freedom:   {dof}')
print(f'P-value:              {p_value:.4f}')
print()

if p_value < 0.05:
    print('✅ p < 0.05 → Variables are NOT independent')
    print('   Promotion rate differs by department.')
else:
    print('❌ p ≥ 0.05 → Cannot conclude dependence')
    print('   Promotion rate may be similar across departments.')

In [ ]:
# ANOVA: Compare means across multiple groups
print('=== ANOVA: Salary Across All Departments ===')
print('Does salary differ significantly across departments?')
print()

# Group salaries by department
groups = [df[df['department'] == dept]['salary'] for dept in df['department'].unique()]

# One-way ANOVA
f_stat, p_value = stats.f_oneway(*groups)

print(f'F-statistic: {f_stat:.4f}')
print(f'P-value:     {p_value:.4f}')
print()

# Summary by department
dept_summary = df.groupby('department')['salary'].agg(['mean', 'std', 'count']).round(0)
print('Department Summary:')
print(dept_summary)
print()

if p_value < 0.05:
    print('✅ p < 0.05 → At least one group mean differs significantly')
else:
    print('❌ p ≥ 0.05 → No significant difference across groups')

---

## 6. Correlation and Covariance

Measuring relationships between variables is crucial for feature selection and understanding data.

### Covariance
Measures how two variables change together:
$$Cov(X, Y) = \frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})$$

### Correlation (Pearson)
Standardized covariance (range: -1 to 1):
$$r = \frac{Cov(X, Y)}{\sigma_X \sigma_Y}$$

| Correlation | Strength | Interpretation |
|-------------|----------|---------------|
| 0.0 - 0.3 | Weak | Little linear relationship |
| 0.3 - 0.7 | Moderate | Some linear relationship |
| 0.7 - 1.0 | Strong | Strong linear relationship |

In [ ]:
# Calculate correlation matrix
numeric_cols = ['age', 'salary', 'experience_years', 'performance_score', 'satisfaction']
corr_matrix = df[numeric_cols].corr()

print('=== Pearson Correlation Matrix ===')
print(corr_matrix.round(3))

In [ ]:
# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1, cbar_kws={'label': 'Correlation'})

ax.set_title('Correlation Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

# Find highest correlations
print('\nHighest Correlations (excluding self):')
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr = corr_matrix.iloc[i, j]
        if abs(corr) > 0.3:
            print(f'  {corr_matrix.columns[i]:20s} ↔ {corr_matrix.columns[j]:20s}: r = {corr:.3f}')

In [ ]:
# Correlation types visualization
np.random.seed(42)
n = 100

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Strong positive correlation
x = np.random.normal(0, 1, n)
y_pos = x + np.random.normal(0, 0.3, n)
r_pos = np.corrcoef(x, y_pos)[0, 1]
axes[0].scatter(x, y_pos, alpha=0.6)
axes[0].set_title(f'Strong Positive\nr = {r_pos:.2f}', fontweight='bold')

# Moderate positive correlation
y_mod = 0.5*x + np.random.normal(0, 0.8, n)
r_mod = np.corrcoef(x, y_mod)[0, 1]
axes[1].scatter(x, y_mod, alpha=0.6)
axes[1].set_title(f'Moderate Positive\nr = {r_mod:.2f}', fontweight='bold')

# No correlation
y_none = np.random.normal(0, 1, n)
r_none = np.corrcoef(x, y_none)[0, 1]
axes[2].scatter(x, y_none, alpha=0.6)
axes[2].set_title(f'No Correlation\nr = {r_none:.2f}', fontweight='bold')

# Strong negative correlation
y_neg = -x + np.random.normal(0, 0.3, n)
r_neg = np.corrcoef(x, y_neg)[0, 1]
axes[3].scatter(x, y_neg, alpha=0.6)
axes[3].set_title(f'Strong Negative\nr = {r_neg:.2f}', fontweight='bold')

for ax in axes:
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation does NOT imply causation!
# And correlation only captures LINEAR relationships

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Non-linear relationship (quadratic)
x = np.linspace(-3, 3, 100)
y_quad = x**2 + np.random.normal(0, 0.5, 100)
r_quad = np.corrcoef(x, y_quad)[0, 1]
axes[0].scatter(x, y_quad, alpha=0.6)
axes[0].set_title(f'Quadratic Relationship\nPearson r = {r_quad:.2f} (misleading!)', fontweight='bold')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')

# Sinusoidal relationship
y_sin = np.sin(x) + np.random.normal(0, 0.2, 100)
r_sin = np.corrcoef(x, y_sin)[0, 1]
axes[1].scatter(x, y_sin, alpha=0.6)
axes[1].set_title(f'Sinusoidal Relationship\nPearson r = {r_sin:.2f} (misleading!)', fontweight='bold')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')

# Anscombe's quartet example - same correlation, different patterns
axes[2].text(0.5, 0.5, 'Remember:\n\n• Correlation only measures\n  LINEAR relationships\n\n• Always visualize your data!\n\n• Correlation ≠ Causation',
             transform=axes[2].transAxes, fontsize=12, ha='center', va='center',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[2].axis('off')

plt.tight_layout()
plt.show()

print('💡 ML Tip: Use Spearman correlation for monotonic (non-linear) relationships.')

In [ ]:
# Spearman correlation (rank-based, captures monotonic relationships)
print('=== Pearson vs Spearman Correlation ===')
print()

# Compare for our data
for col1, col2 in [('experience_years', 'salary'), ('age', 'satisfaction')]:
    pearson_r, pearson_p = stats.pearsonr(df[col1], df[col2])
    spearman_r, spearman_p = stats.spearmanr(df[col1], df[col2])
    
    print(f'{col1} vs {col2}:')
    print(f'  Pearson:  r={pearson_r:.3f}, p={pearson_p:.4f}')
    print(f'  Spearman: r={spearman_r:.3f}, p={spearman_p:.4f}')
    print()

print('💡 When to use which:')
print('  • Pearson: Linear relationships, normally distributed data')
print('  • Spearman: Monotonic relationships, ordinal data, outliers present')

---

## 7. Practice Exercises

### Exercise 1: Descriptive Statistics

For the `performance_score` column:
1. Calculate mean, median, and mode
2. Calculate standard deviation and IQR
3. Calculate skewness and interpret it
4. Is the distribution approximately normal? (use Shapiro-Wilk test)

In [ ]:
# Exercise 1: Your solution here


### Exercise 2: Hypothesis Testing

1. Test if employees with high satisfaction (above median) have different performance scores than those with low satisfaction
2. Use an independent samples t-test
3. Calculate the effect size (Cohen's d)
4. Interpret both statistical and practical significance

In [ ]:
# Exercise 2: Your solution here


### Exercise 3: Probability

Using the dataset:
1. What's the probability that a randomly selected employee is from Engineering AND was promoted?
2. What's the probability of promotion given the employee is in Marketing?
3. If someone was promoted, what's the probability they're from HR? (use Bayes' theorem)

In [ ]:
# Exercise 3: Your solution here


### Exercise 4: Confidence Intervals & CLT

1. Calculate the 95% confidence interval for mean performance_score
2. Take 100 random samples of size 50, calculate means
3. Plot the distribution of sample means
4. Does it look normal? (demonstrate CLT)

In [ ]:
# Exercise 4: Your solution here


---

## 8. Summary & Quick Reference

### Statistical Tests Cheat Sheet

| Question | Test | Python |
|----------|------|--------|
| Compare 2 means | t-test | `stats.ttest_ind(a, b)` |
| Compare 3+ means | ANOVA | `stats.f_oneway(a, b, c)` |
| Compare proportions | Chi-square | `stats.chi2_contingency(table)` |
| Test normality | Shapiro-Wilk | `stats.shapiro(x)` |
| Linear correlation | Pearson | `stats.pearsonr(x, y)` |
| Monotonic correlation | Spearman | `stats.spearmanr(x, y)` |

### Key Formulas

| Measure | Formula |
|---------|--------|
| Mean | $\bar{x} = \frac{1}{n}\sum x_i$ |
| Variance | $\sigma^2 = \frac{1}{n}\sum(x_i - \bar{x})^2$ |
| Standard Error | $SE = \frac{\sigma}{\sqrt{n}}$ |
| Z-score | $z = \frac{x - \mu}{\sigma}$ |
| 95% CI | $\bar{x} \pm 1.96 \cdot SE$ |
| Cohen's d | $d = \frac{\bar{x}_1 - \bar{x}_2}{s_{pooled}}$ |

### Decision Rules

| P-value | Interpretation |
|---------|---------------|
| p < 0.001 | Very strong evidence against H₀ |
| p < 0.01 | Strong evidence against H₀ |
| p < 0.05 | Moderate evidence against H₀ |
| p ≥ 0.05 | Insufficient evidence to reject H₀ |

In [ ]:
# Quick reference code snippets
print('📋 Statistics Quick Reference Code')
print('=' * 50)
print()
print('# Descriptive Statistics')
print('df.describe()  # All at once')
print('df["col"].mean(), df["col"].median(), df["col"].std()')
print('df["col"].skew(), df["col"].kurtosis()')
print()
print('# Normality Test')
print('stats.shapiro(data)  # Returns (statistic, p-value)')
print()
print('# T-test (compare 2 groups)')
print('stats.ttest_ind(group1, group2)')
print()
print('# ANOVA (compare 3+ groups)')
print('stats.f_oneway(group1, group2, group3)')
print()
print('# Correlation')
print('df.corr()  # Correlation matrix')
print('stats.pearsonr(x, y)  # Pearson r and p-value')
print('stats.spearmanr(x, y)  # Spearman rho and p-value')
print()
print('# Confidence Interval')
print('se = data.std() / np.sqrt(len(data))')
print('ci = (mean - 1.96*se, mean + 1.96*se)')

---

## Next Steps

You've completed the Statistics for ML module! You now know how to:

✅ Calculate and interpret descriptive statistics  
✅ Apply probability rules and Bayes' theorem  
✅ Understand key probability distributions  
✅ Use the Central Limit Theorem for inference  
✅ Conduct hypothesis tests (t-test, ANOVA, chi-square)  
✅ Measure and interpret correlation  

**Continue to the next notebook:**
- `05_scikit_learn_introduction.ipynb` - Your first ML models!